# Bone suppression for chest X-rays on CPU (qureaiorg/bone-suppression)

Runs the Qure.ai bone and lung-component suppression TorchScript models ([HF model](https://huggingface.co/qureaiorg/bone-suppression), [arXiv:2609.24937](https://arxiv.org/abs/2609.24937)) and the pre-registered stability experiment from [`apps/bone-suppression-cxr`](https://github.com/Ryukijano/tech-demos/tree/main/apps/bone-suppression-cxr).

* **Runs on the free CPU runtime.** Expect about 5 s per model per 1024 px image on a few CPU cores (Colab's 2-core CPU will be slower). A GPU is optional. It is only used if you switch the runtime yourself, and on the free tier it costs nothing.
* Weights are **CC BY-NC-SA 4.0** (non-commercial research and education only) and are downloaded from Hugging Face at run time. No token is needed.
* **Research use only. This is not a medical device and must not be used for diagnosis.**

In [ ]:
!pip -q install opencv-python-headless scikit-image huggingface_hub pyyaml pandas matplotlib
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/Ryukijano/tech-demos.git
%cd tech-demos
!git sparse-checkout set apps/bone-suppression-cxr
%cd apps/bone-suppression-cxr

## 1. Decompose one image (bone / soft tissue / lung component / non-lung soft tissue)

In [ ]:
import sys, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, 'src'); import bsx
x = bsx.preprocess(bsx.read_gray(bsx.fetch('hf-space:hugging-apps/bone-suppression-chest-xray/examples/cxr_train_1_a_1.png')))
bone_m = bsx.load_model('bone'); bone, t1 = bsx.run(bone_m, x); soft = x - bone
lung_m = bsx.load_model('lung'); lung, t2 = bsx.run(lung_m, soft)   # float residual, no re-normalisation
print(f'bone {t1:.1f}s  lung {t2:.1f}s  (first call includes TorchScript warm-up)')
ims = [('input', bsx.to_u8(x)), ('bone', bsx.to_u8(bone, True)), ('soft = full - bone', bsx.to_u8(soft)),
       ('lung component', bsx.to_u8(lung, True)), ('non-lung soft', bsx.to_u8(soft - lung))]
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
for a, (n, im) in zip(ax, ims): a.imshow(im, cmap='gray'); a.set_title(n); a.axis('off')

## 2. Full pre-registered experiment (seeds 0/1/2 + breakpoints)
The thresholds are fixed in `experiments/exp1_cpu_stability/config.yaml`. A full CPU run is about 90 model passes. Add `--skip-breakpoints` for a quicker run.

In [ ]:
del bone_m, lung_m
!python src/run_experiment.py
!python src/analyze.py
!cat experiments/exp1_cpu_stability/results.md

In [ ]:
from IPython.display import Image, display
display(Image('experiments/exp1_cpu_stability/plot.png'))
!cat experiments/exp1_cpu_stability/breakpoints.md